# MINE: Combined Analysis, Three Regimes

Loads results from the three model notebooks and produces:
- 4-panel figure
- Diagnostic verdict
- LaTeX table

No GPU required. Reads results JSON files from `./results/mine/`.

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd

PHASE = 'full'
OUTPUT_BASE = './results/mine_v2_three_regimes/'
RESULTS_DIR = OUTPUT_BASE + 'results'

# Load all result JSONs
all_results = []
all_traces = {}
procrustes_results = []

for tag in ['protmamba', 'openfold', 'evo2']:
    rfile = f'{RESULTS_DIR}/{tag}_results_{PHASE}.json'
    tfile = f'{RESULTS_DIR}/{tag}_traces_{PHASE}.json'
    pfile = f'{RESULTS_DIR}/{tag}_procrustes_{PHASE}.json'

    if os.path.exists(rfile):
        with open(rfile) as f:
            results = json.load(f)
        all_results.extend(results)
        print(f'Loaded {tag}: {len(results)} result rows')
    else:
        print(f'MISSING: {rfile}')

    if os.path.exists(tfile):
        with open(tfile) as f:
            traces = json.load(f)
        all_traces.update(traces)
        print(f'  + {len(traces)} trace sets')

    if os.path.exists(pfile):
        with open(pfile) as f:
            proc = json.load(f)
        procrustes_results.extend(proc)
        print(f'  + {len(proc)} Procrustes results')

df_mine = pd.DataFrame(all_results)
print(f'\nTotal rows: {len(df_mine)}')
print(df_mine[['model', 'regime', 'seq_length', 'mi_mean', 'mi_std']].to_string(index=False))

In [ ]:
# ============================================================
# Bias Correction: Excess MI over matched random baseline
# ============================================================
# MINE has finite-sample bias that scales with input dimensionality.
# Standard fix: report MI_model - MI_random (excess over noise floor).
# Negative excess = model embeddings have LESS biological structure than noise.

random_baselines = {}
for _, row in df_mine[df_mine['model'].str.startswith('Random')].iterrows():
    gt = row['ground_truth']
    random_baselines[gt] = row['mi_mean']

print("Random baselines (MINE bias floor):")
for gt, val in random_baselines.items():
    print(f"  {gt}: {val:.4f} nats")

def get_excess(row):
    gt = row.get('ground_truth', '')
    if gt in ('dna_local',):
        gt = 'dna'
    baseline = random_baselines.get(gt, 0)
    return row['mi_mean'] - baseline

df_mine['mi_excess'] = df_mine.apply(get_excess, axis=1)

print("\nExcess MI (bias-corrected):")
cols = ['model', 'regime', 'mi_mean', 'mi_excess']
print(df_mine[~df_mine['model'].str.startswith('Random')][cols].to_string(index=False))

## Visualization: The Three Regimes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.rcParams.update({'font.size': 11, 'font.family': 'sans-serif'})

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

# Noise floors
rand_prot = df_mine[df_mine['model']=='Random_Protein']['mi_mean'].values
noise_floor_protein = rand_prot[0] + 2 * df_mine[df_mine['model']=='Random_Protein']['mi_std'].values[0] if len(rand_prot) else 0

rand_dna = df_mine[df_mine['model']=='Random_DNA']['mi_mean'].values
noise_floor_dna = rand_dna[0] + 2 * df_mine[df_mine['model']=='Random_DNA']['mi_std'].values[0] if len(rand_dna) else 0


# ======== Panel A: Regime Bar Chart (normalized MI) ========
ax = axes[0, 0]
regime_models = df_mine[~df_mine['model'].str.startswith('Random') &
                        ~df_mine['model'].str.startswith('Evo2_Local')].copy()

COLOR_MAP = {
    'ProtMamba': '#EF4444',
    'OpenFold': '#F59E0B',
    'ESM1b_Raw': '#3B82F6',
    'Evo2_Global': '#7C3AED',
}

bar_labels = []
bar_values = []
bar_colors = []
bar_errs = []

for _, row in regime_models.iterrows():
    model = row['model']
    if model in ('Random_Protein', 'Random_DNA'):
        continue
    if row.get('seq_length', 0) > 0 and model not in ('Evo2_Global',):
        lbl = f"{model}\nL={int(row['seq_length'])}"
    elif model == 'Evo2_Global':
        lbl = f"Evo 2 (7B)\nGlobal"
    else:
        lbl = model
    bar_labels.append(lbl)

    if 'mi_normalized' in row and not np.isnan(row.get('mi_normalized', np.nan)):
        bar_values.append(row['mi_normalized'])
        bar_errs.append(row['mi_std'] / max(row.get('mi_ceiling', 1), 1e-8))
    else:
        bar_values.append(row['mi_mean'])
        bar_errs.append(row['mi_std'])
    bar_colors.append(COLOR_MAP.get(model, '#6B7280'))

bars = ax.bar(range(len(bar_labels)), bar_values, yerr=bar_errs,
              color=bar_colors, alpha=0.85, capsize=5, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(bar_labels)))
ax.set_xticklabels(bar_labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Normalized MI (fraction of ceiling)', fontsize=10)
ax.set_title('A. Normalized MI Across Regimes', fontweight='bold', fontsize=12)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.15, axis='y')

for bar, val in zip(bars, bar_values):
    ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 8), textcoords='offset points', ha='center', fontsize=7,
                fontweight='bold')


# ======== Panel B: Training Traces ========
ax = axes[0, 1]
TRACE_COLORS = {
    'ProtMamba': '#EF4444', 'OpenFold': '#F59E0B', 'ESM1b_Raw': '#3B82F6',
    'Evo2': '#7C3AED', 'Random': '#9CA3AF',
}

for key, trc in all_traces.items():
    traces_arr = np.array(trc)
    mean_trace = traces_arr.mean(axis=0)
    std_trace = traces_arr.std(axis=0)
    epochs = np.arange(1, len(mean_trace)+1)
    color = '#6B7280'
    for prefix, c in TRACE_COLORS.items():
        if key.startswith(prefix):
            color = c
            break
    ax.plot(epochs, mean_trace, color=color, linewidth=1.2, label=key, alpha=0.7)
    ax.fill_between(epochs, mean_trace - std_trace, mean_trace + std_trace,
                    color=color, alpha=0.08)

ax.set_xlabel('Epoch', fontsize=10)
ax.set_ylabel('MI Lower Bound (nats)', fontsize=10)
ax.set_title('B. MINE Training Convergence', fontweight='bold', fontsize=12)
ax.legend(fontsize=6, loc='upper left', ncol=2)
ax.grid(True, alpha=0.15)
ax.axhline(y=0, color='black', linestyle=':', linewidth=0.8)


# ======== Panel C: Local vs Global MI (Regime I) ========
ax = axes[1, 0]
local_rows = df_mine[df_mine['model'].str.startswith('Evo2_Local')]
global_rows = df_mine[df_mine['model'] == 'Evo2_Global']

if len(local_rows) > 0 and len(global_rows) > 0:
    global_mi = global_rows.iloc[0]['mi_mean']
    global_std = global_rows.iloc[0]['mi_std']

    categories = ['Global\n(mean-pool)']
    values = [global_mi]
    errs = [global_std]
    colors_c = ['#7C3AED']

    for _, row in local_rows.iterrows():
        w = row.get('window', '')
        categories.append(f'Local\n[{w}]')
        values.append(row['mi_mean'])
        errs.append(row['mi_std'])
        colors_c.append('#A78BFA')

    bars_c = ax.bar(range(len(categories)), values, yerr=errs,
                    color=colors_c, alpha=0.85, capsize=5, edgecolor='white', linewidth=1.5)
    ax.set_xticks(range(len(categories)))
    ax.set_xticklabels(categories, fontsize=8)
    ax.axhline(y=noise_floor_dna, color='#9CA3AF', linestyle='--', linewidth=1.2,
               label=f'DNA noise floor ({noise_floor_dna:.3f})')
    ax.set_ylabel('MI (nats)', fontsize=10)
    ax.set_title('C. Regime I: Local-Global Decoupling (Evo 2)', fontweight='bold', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15, axis='y')

    for bar, val in zip(bars_c, values):
        ax.annotate(f'{val:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 8), textcoords='offset points', ha='center', fontsize=8,
                    fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Evo 2 not available', transform=ax.transAxes,
            ha='center', va='center', fontsize=12, color='gray')
    ax.set_title('C. Regime I: Local-Global Decoupling', fontweight='bold', fontsize=12)


# ======== Panel D: Regime II (ESM-1b -> Evoformer + Procrustes) ========
ax = axes[1, 1]

esm_rows = df_mine[df_mine['model'] == 'ESM1b_Raw'].sort_values('seq_length')
evo_rows = df_mine[df_mine['model'] == 'OpenFold'].sort_values('seq_length')

if len(esm_rows) > 0 and len(evo_rows) > 0:
    x_pos = np.arange(len(esm_rows))
    width = 0.35

    ax.bar(x_pos - width/2, esm_rows['mi_mean'].values, width, yerr=esm_rows['mi_std'].values,
           label='ESM-1b (pre-Evoformer)', color='#3B82F6', alpha=0.85, capsize=4)
    ax.bar(x_pos + width/2, evo_rows['mi_mean'].values, width, yerr=evo_rows['mi_std'].values,
           label='Evoformer (post)', color='#F59E0B', alpha=0.85, capsize=4)

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'L={int(v)}' for v in esm_rows['seq_length'].values], fontsize=9)
    ax.axhline(y=noise_floor_protein, color='#9CA3AF', linestyle='--', linewidth=1.2,
               label=f'Protein noise floor ({noise_floor_protein:.3f})')

    if procrustes_results:
        for i, pr in enumerate(procrustes_results):
            if i < len(x_pos):
                ax.annotate(f"Procrustes\ndisp={pr['disparity']:.4f}",
                           xy=(x_pos[i], max(esm_rows['mi_mean'].values[i],
                                             evo_rows['mi_mean'].values[i])),
                           xytext=(0, 20), textcoords='offset points',
                           ha='center', fontsize=7, color='#DC2626',
                           fontweight='bold')

    ax.set_ylabel('MI (nats)', fontsize=10)
    ax.set_title('D. Regime II: ESM-1b vs Evoformer (OpenFold)', fontweight='bold', fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15, axis='y')

fig.suptitle('MINE: Three Regimes of the Geometric Alignment Tax',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mine_three_regimes_{PHASE}.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.savefig(f'{RESULTS_DIR}/mine_three_regimes_{PHASE}.pdf', bbox_inches='tight')
plt.show()
print(f'Saved: {RESULTS_DIR}/mine_three_regimes_{PHASE}.png')

## Diagnostic Verdict

In [ ]:
# Extract ceilings from stored results
mi_ceiling_protein = df_mine[df_mine['ground_truth']=='protein']['mi_ceiling'].dropna().iloc[0] if len(df_mine[df_mine['ground_truth']=='protein']['mi_ceiling'].dropna()) else 1.0
mi_ceiling_dna_vals = df_mine[df_mine['ground_truth']=='dna']['mi_ceiling'].dropna()
mi_ceiling_dna = mi_ceiling_dna_vals.iloc[0] if len(mi_ceiling_dna_vals) else None

# Noise floors
rand_prot_rows = df_mine[df_mine['model']=='Random_Protein']
rand_dna_rows = df_mine[df_mine['model']=='Random_DNA']

nf_prot = (rand_prot_rows.iloc[0]['mi_mean'] + 2 * rand_prot_rows.iloc[0]['mi_std']) if len(rand_prot_rows) else 0.0
nf_dna = (rand_dna_rows.iloc[0]['mi_mean'] + 2 * rand_dna_rows.iloc[0]['mi_std']) if len(rand_dna_rows) else 0.0

print('=' * 70)
print('THREE-REGIME DIAGNOSTIC VERDICT')
print('=' * 70)

print(f'\nProtein noise floor: {nf_prot:.4f} nats')
print(f'DNA noise floor: {nf_dna:.4f} nats')
print(f'Protein MI ceiling: {mi_ceiling_protein:.4f} nats')
if mi_ceiling_dna is not None:
    print(f'DNA MI ceiling: {mi_ceiling_dna:.4f} nats')

# Regime III: ProtMamba
pm_rows = df_mine[df_mine['model'] == 'ProtMamba']
if len(pm_rows):
    print(f'\n{"="*50}')
    print('REGIME III: ProtMamba (Geometric Vacuity)')
    print('='*50)
    ppl = pm_rows.iloc[0].get('perplexity', None)
    for _, row in pm_rows.iterrows():
        mi = row['mi_mean']
        norm = row.get('mi_normalized', mi / max(mi_ceiling_protein, 1e-8))
        below = mi <= nf_prot
        print(f'  L={int(row["seq_length"])}: MI={mi:.4f} nats '
              f'(normalized={norm:.4f}, {"BELOW" if below else "above"} noise floor)')
    if ppl is not None:
        print(f'  Task perplexity: {ppl:.2f}')
        if below and ppl is not None and ppl < 50:
            print(f'  CONFIRMED: Reasonable perplexity ({ppl:.1f}) + noise-floor MI')
            print(f'  = Geometric Vacuity (smooth but informationally empty)')

# Regime II: OpenFold
of_rows = df_mine[df_mine['model'] == 'OpenFold']
esm_rows = df_mine[df_mine['model'] == 'ESM1b_Raw']
if len(of_rows) and len(esm_rows):
    print(f'\n{"="*50}')
    print('REGIME II: OpenFold (Representational Compression)')
    print('='*50)
    for _, of_row in of_rows.iterrows():
        sl = int(of_row['seq_length'])
        esm_match = esm_rows[esm_rows['seq_length'] == sl]
        if len(esm_match):
            esm_mi = esm_match.iloc[0]['mi_mean']
            evo_mi = of_row['mi_mean']
            delta = evo_mi - esm_mi
            pct = (delta / esm_mi * 100) if esm_mi > 0 else 0
            proc = [p for p in procrustes_results if p['seq_length'] == sl]
            disp = proc[0]['disparity'] if proc else float('nan')
            print(f'  L={sl}: ESM-1b={esm_mi:.3f} -> Evoformer={evo_mi:.3f} '
                  f'(delta={delta:+.3f}, {pct:+.1f}%)')
            print(f'         Procrustes disparity: {disp:.6f}')

# Regime I: Evo 2
evo_global = df_mine[df_mine['model'] == 'Evo2_Global']
evo_local = df_mine[df_mine['model'].str.startswith('Evo2_Local')]
if len(evo_global) and len(evo_local):
    print(f'\n{"="*50}')
    print('REGIME I: Evo 2 (Local-Global Decoupling)')
    print('='*50)
    gmi = evo_global.iloc[0]['mi_mean']
    lmi_vals = evo_local['mi_mean'].values
    avg_lmi = np.mean(lmi_vals)
    print(f'  Global MI: {gmi:.4f} nats ({"BELOW" if gmi <= nf_dna else "above"} noise floor)')
    print(f'  Local MI (mean): {avg_lmi:.4f} nats')
    for _, row in evo_local.iterrows():
        w = row.get('window', '?')
        print(f'    Window [{w}]: {row["mi_mean"]:.4f} nats')

print(f'\n{"="*70}')
print('COMPLETE RESULTS TABLE')
print('='*70)
cols = ['model', 'regime', 'seq_length', 'mi_mean', 'mi_std']
if 'mi_normalized' in df_mine.columns:
    cols.append('mi_normalized')
print(df_mine[cols].to_string(index=False))

## LaTeX Table

In [ ]:
print(r'\begin{table}[H]')
print(r'\centering')
print(r'\caption{\textbf{MINE Mutual Information Estimates: Three Regimes of the Geometric')
print(r'Alignment Tax.} Donsker-Varadhan lower bound on $I(X; \hat{X})$ between biological')
print(r'ground truth $X$ and model embeddings $\hat{X}$. Normalized MI = raw MI / ceiling.')
print(r'Per-distribution noise floors and ceilings ensure cross-model comparability.}')
print(r'\label{tab:mine_three_regimes}')
print(r'\begin{tabular}{llccccc}')
print(r'\toprule')
print(r'\textbf{Regime} & \textbf{Model} & \textbf{Scope} & '
      r'\textbf{$\hat{I}$} & \textbf{$\pm$std} & '
      r'\textbf{Norm.} & \textbf{Verdict} \\')
print(r'\midrule')

# Regime III
pm_rows_tex = df_mine[df_mine['model'] == 'ProtMamba']
for _, row in pm_rows_tex.iterrows():
    mi = row['mi_mean']
    norm = row.get('mi_normalized', 0)
    verdict = r'\textcolor{red}{Vacuity}'
    print(f'  III & ProtMamba (L={int(row["seq_length"])}) & Global & '
          f'{mi:.4f} & {row["mi_std"]:.4f} & {norm:.4f} & {verdict} \\\\')

print(r'\midrule')

# Regime II
for _, row in df_mine[df_mine['model'].isin(['ESM1b_Raw', 'OpenFold'])].sort_values(
        ['seq_length', 'model']).iterrows():
    mi = row['mi_mean']
    norm = row.get('mi_normalized', 0)
    if row['model'] == 'ESM1b_Raw':
        verdict = r'\textcolor{blue}{Baseline}'
        name = 'ESM-1b (pre)'
    else:
        verdict = r'\textcolor{orange}{Compression}'
        name = 'Evoformer (post)'
    print(f'  II & {name} (L={int(row["seq_length"])}) & Global & '
          f'{mi:.4f} & {row["mi_std"]:.4f} & {norm:.4f} & {verdict} \\\\')

print(r'\midrule')

# Regime I
evo_g = df_mine[df_mine['model'] == 'Evo2_Global']
evo_l = df_mine[df_mine['model'].str.startswith('Evo2_Local')]
if len(evo_g):
    row = evo_g.iloc[0]
    mi = row['mi_mean']
    norm = row.get('mi_normalized', 0)
    print(f'  I & Evo 2 (7B) & Global & '
          f'{mi:.4f} & {row["mi_std"]:.4f} & {norm:.4f} & '
          r'\textcolor{red}{Ghost} \\')
for _, row in evo_l.iterrows():
    mi = row['mi_mean']
    norm = row.get('mi_normalized', 0)
    w = row.get('window', '?')
    print(f'  I & Evo 2 (7B) & Local [{w}] & '
          f'{mi:.4f} & {row["mi_std"]:.4f} & {norm:.4f} & '
          r'\textcolor{teal}{Signal} \\')

print(r'\midrule')

# Baselines
for _, row in df_mine[df_mine['model'].str.startswith('Random')].iterrows():
    mi = row['mi_mean']
    gt = row.get('ground_truth', '?')
    print(f'  -- & Random ({gt}) & -- & '
          f'{mi:.4f} & {row["mi_std"]:.4f} & -- & '
          r'\textcolor{gray}{Floor} \\')

print(r'\bottomrule')
print(r'\end{tabular}')
print(r'\end{table}')